In [1]:
import torch

In [3]:
A = torch.ones(8, 16, 1024)
B = torch.ones(8, 16, 1024)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LanguageConditionedCrossAttention(nn.Module):
    def __init__(self, obj_dim, lang_dim, hidden_dim, num_heads=4):
        super(LanguageConditionedCrossAttention, self).__init__()
        self.obj_dim = obj_dim
        self.lang_dim = lang_dim
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        assert self.hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        # Object projections
        self.q_proj = nn.Linear(obj_dim, hidden_dim)
        self.k_proj = nn.Linear(obj_dim, hidden_dim)
        self.v_proj = nn.Linear(obj_dim, hidden_dim)

        # Language-conditioned bias terms
        self.q_lang_proj = nn.Linear(lang_dim, hidden_dim)
        self.k_lang_proj = nn.Linear(lang_dim, hidden_dim)
        self.v_lang_proj = nn.Linear(lang_dim, hidden_dim)

        # Output projection
        self.out_proj = nn.Linear(hidden_dim, obj_dim)

    def forward(self, obj_feats, lang_embed):
        """
        obj_feats: (B, N, D)       - object features (batch, num_objects, obj_dim)
        lang_embed: (B, L)         - sentence embedding (batch, lang_dim)
        """
        B, N, _ = obj_feats.shape

        # Project language to same dim as hidden_dim
        q_lang = self.q_lang_proj(lang_embed).unsqueeze(1)  # (B, 1, H)
        k_lang = self.k_lang_proj(lang_embed).unsqueeze(1)
        v_lang = self.v_lang_proj(lang_embed).unsqueeze(1)

        # Project objects
        Q = self.q_proj(obj_feats) + q_lang   # (B, N, H)
        K = self.k_proj(obj_feats) + k_lang
        V = self.v_proj(obj_feats) + v_lang

        # Reshape for multi-head attention
        def reshape(x):
            return x.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
            # (B, num_heads, N, head_dim)

        Q = reshape(Q)
        K = reshape(K)
        V = reshape(V)

        # Scaled dot-product attention
        attn_logits = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, heads, N, N)
        attn_weights = F.softmax(attn_logits, dim=-1)  # (B, heads, N, N)

        attended = torch.matmul(attn_weights, V)  # (B, heads, N, head_dim)

        # Combine heads
        attended = attended.transpose(1, 2).contiguous().view(B, N, self.hidden_dim)  # (B, N, H)

        # Project back to object feature dim
        out = self.out_proj(attended)  # (B, N, obj_dim)

        return out  # language-modulated object features


In [5]:
B = 2           # Batch size
N = 5           # Number of objects
D_obj = 128     # Object feature dimension
D_lang = 256    # Language embedding dimension

obj_feats = torch.randn(B, N, D_obj)
lang_embed = torch.randn(B, D_lang)

attn_layer = LanguageConditionedCrossAttention(obj_dim=D_obj, lang_dim=D_lang, hidden_dim=256)
out_feats = attn_layer(obj_feats, lang_embed)

print(out_feats.shape) 

torch.Size([2, 5, 128])


In [7]:
class TargetToRelationalCrossAttention(nn.Module):
    def __init__(self, target_dim, relational_dim, lang_dim, hidden_dim, num_heads=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        assert hidden_dim % num_heads == 0

        # Projections
        self.q_proj = nn.Linear(target_dim, hidden_dim)
        self.k_proj = nn.Linear(relational_dim, hidden_dim)
        self.v_proj = nn.Linear(relational_dim, hidden_dim)

        # Language modulation
        self.q_lang_proj = nn.Linear(lang_dim, hidden_dim)
        self.k_lang_proj = nn.Linear(lang_dim, hidden_dim)
        self.v_lang_proj = nn.Linear(lang_dim, hidden_dim)

        self.out_proj = nn.Linear(hidden_dim, target_dim)

    def forward(self, targets, relationals, lang_feat):
        """
        targets:     (B, N, d_t) — N target candidate features
        relationals: (B, N, d_r) — N relational object features
        lang_feat:   (B, d_l)    — language sentence embedding
        Returns:
            updated_targets: (B, N, d_t)
        """
        B, N, _ = targets.shape

        # Project language
        q_lang = self.q_lang_proj(lang_feat).unsqueeze(1)  # (B, 1, H)
        k_lang = self.k_lang_proj(lang_feat).unsqueeze(1)
        v_lang = self.v_lang_proj(lang_feat).unsqueeze(1)

        # Project inputs
        Q = self.q_proj(targets) + q_lang  # (B, N, H)
        K = self.k_proj(relationals) + k_lang
        V = self.v_proj(relationals) + v_lang

        # Reshape for multi-head
        def reshape(x):
            return x.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

        Q = reshape(Q)  # (B, heads, N, head_dim)
        K = reshape(K)
        V = reshape(V)

        # Attention: Q from targets, K/V from relational objects
        attn_logits = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, heads, N, N)
        attn_weights = F.softmax(attn_logits, dim=-1)
        attended = torch.matmul(attn_weights, V)  # (B, heads, N, head_dim)

        # Merge heads
        attended = attended.transpose(1, 2).contiguous().view(B, N, self.hidden_dim)  # (B, N, H)

        # Final projection
        updated_targets = self.out_proj(attended)  # (B, N, d_t)

        return updated_targets

In [8]:
B, N = 2, 5
d_t, d_r, d_l = 128, 128, 256

targets = torch.randn(B, N, d_t)
rel_objs = torch.randn(B, N, d_r)
lang = torch.randn(B, d_l)

cross_attn = TargetToRelationalCrossAttention(d_t, d_r, d_l, hidden_dim=256)
updated_targets = cross_attn(targets, rel_objs, lang)

print(updated_targets.shape)  # (B, N, d_t)

torch.Size([2, 5, 128])
